# Custom Log4J Logger Wrapper object Class
- Should be used to log custom informational, error or warning messages to the Log4J log in Databricks
- Messages should guide the person troubleshooting the code if something should fail.

In [82]:
# Databricks notebook source
from pyspark.context import SparkContext
from pyspark.sql import SparkSession
from typing import Optional
import logging
import yaml
import pathlib
import os
#from pyspark import SparkConf, SparkContext


In [83]:
class Log4jWrapper(object):
    """
    ------------------------------
    Description
    ------------------------------
    A class that creates a callable instance of the PySpark built in Log4j JVM logging object.
    Log4j is a Java-based logging utility part of the Apache Logging Services. 
    
    ------------------------------
    Parameters
    ------------------------------
    spark:
        SparkSession object.
    """

    def __init__(self, spark, spark_conf):
        """        
        ------------------------------
        Description
        ------------------------------
        Initialise the class with configuration varaibles and generate the custom log4j logger object.
        Will create the message prefix standard value to be used when logging errors, warnings or informational messages in this format.
            'Custom Message Logged from <applicationName: <ApplicationName> | applicationID: <ID> | notebookName: <NotebookName>>
        
        ------------------------------
        Parameters
        ------------------------------
        spark:
            The currently active SparkSession object        
            
        ------------------------------
        Return Value(s)
        ------------------------------
        None
        
        ------------------------------
        Example usage
        ------------------------------
        vLog4jWrapper = Log4jWrapper(spark)

        vLog4jWrapper.info('Test info logging')
        vLog4jWrapper.error('Test error logging')
        vLog4jWrapper.warning('Test warning logging')
        
        """
        # Get the currently active Spark Application details with which to prefix all messages. Store the details in variable(s) that other methods in this class can utilise
        self.sparkConfiguration = spark_conf
        self.applicationID = self.sparkConfiguration.get('spark.master')
        self.applicationName = self.sparkConfiguration.get('spark.app.name')
        
        # Get the notebook name where the class is being instantiated in. Store the details in variables that other methods in this class can utilise
        #self.notebookName = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        self.notebookName = 'emile'

        #Create the custom message prefix to attach to all messages sent to the methods of this class. Store the details in variables that other methods in this class can utilise
        self.message_prefix = 'Custom Message Logged from <applicationName: ' + self.applicationName #+ ' | applicationID: ' + self.applicationID + ' | notebookName: ' + self.notebookName + '>: '
        
        #Get the reference to the Log4h object in this spark application
        log4j = spark._jvm.org.apache.log4j
        
        # Using the Log4j object, get an instance of the logger, passing in the custom message prefix defined above
        self.message_prefix = '<' + self.applicationName #+ ' ' + self.applicationID + ' ' + self.notebookName + '>'
        
        #Create the logger object instance used by all methods of this class
        self.logger = log4j.LogManager.getLogger(self.message_prefix)

    def error(self, message):
        """
        ------------------------------
        Description
        ------------------------------
        Log an error to the Log4j log using the logger defined int the init function.
        The error message should be clear as to what is most likely causing the problem. 
        Provide as much context to the current state of the application as possible to help guide troubleshooting. 
        
        ------------------------------
        Parameters
        ------------------------------
        message:
            The error message to log that describes the current state of the code, data, variables etc. 
            
        ------------------------------
        Return Value(s)
        ------------------------------
        None        
        """
        
        self.logger.error(message)
        
        return None

    def warning(self, message):
        """
        ------------------------------
        Description
        ------------------------------
        Log a Warning to the Log4j log using the logger defined int the init function.
        Warnings should state when best practices are not being followed which will not necesarily cause errors. 
        
        ------------------------------
        Parameters
        ------------------------------
        message:
            The warning message to log that describes the current state of the code, data, variables etc. 
            
        ------------------------------
        Return Value(s)
        ------------------------------
        None        
        """        

        self.logger.warn(message)
        
        return None

    def info(self, message):
        """
        ------------------------------
        Description
        ------------------------------
        Log an informational message to the Log4j log using the logger defined int the init function.
        Typically used for debugging/troubleshooting and general program state logging. 
        Use to log variable values or the current location in the notebook to the Log4J log. 
        
        ------------------------------
        Parameters
        ------------------------------
        message:
            The message to log that describes the current state of the code, data, variables etc. 
            
        ------------------------------
        Return Value(s)
        ------------------------------
        None        
        """            

        self.logger.info(message)
        
        return None

In [84]:
#

sc = SparkContext.getOrCreate()
conf = SparkConf().setMaster("local").setAppName("pyspark-shell")
logger = Log4jWrapper(sc, conf)

logger.info('Test info logging')
logger.error('Test error logging')

23/05/16 01:35:25 ERROR <pyspark-shell: Test error logging
